In [36]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import datasets
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple
from datetime import datetime

from data_process import load_dataset
from processors.demographics import process_demographics, validate_demographics, get_demographic_summary
from processors.measurements import process_bmi, get_bmi_statistics
from processors.cbc import process_cbc, get_cbc_subject_statistics, get_cbc_overall_statistics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Load your data
PATH = '/Users/aashnashah/Desktop/ssh_mount/data/EHRSHOT/meds_omop_ehrshot/'
df = load_dataset(PATH)

In [4]:
# Process demographics
demo_df = process_demographics(df)
display(demo_df.head())
summary = get_demographic_summary(demo_df)
print("Demographics Summary:", summary)

# Save demographics
demo_df.to_csv('data/demographics.csv', index=False)


code,subject_id,Age,Ethnicity,Gender,Race
1,115967096,84,Not Hispanic,F,5
2,115967097,67,Not Hispanic,F,5
3,115967098,81,Not Hispanic,F,5
4,115967099,73,Not Hispanic,F,5
5,115967100,59,Not Hispanic,F,5


Demographics Summary: {'total_subjects': 4735, 'age_stats': {'mean': 62.58627243928194, 'median': 66.0, 'std': 16.78319534247854}, 'gender_distribution': {'F': 2498, 'M': 2237}, 'race_distribution': {'5': 3376, '2': 984, '3': 282, '4': 69, '1': 24}, 'ethnicity_distribution': {'Not Hispanic': 4435, 'Hispanic': 300}}


In [5]:
# only do for subjects with demographics
bmi_df = process_bmi(df[df['subject_id'].isin(demo_df['subject_id'])])
display(bmi_df.head())
summary = get_bmi_statistics(bmi_df)
print("BMI Summary:", summary)

# Save BMI
bmi_df.to_csv('data/bmi.csv', index=False)


Removed 65 invalid BMI values


,subject_id,time,visit_id,BMI_computed,BMI_existing,final_bmi,BMI_category
0,115967096,2009-02-07 00:00:00,32310558,30.335969,30.340000,30.340000,Obesity
1,115967096,2009-02-07 13:20:00,32310558,30.335969,NaN,30.335969,Obesity
2,115967096,2009-06-08 00:00:00,33977166,28.608094,28.610001,28.610001,Overweight
3,115967096,2009-07-04 00:00:00,33720374,28.878704,28.879999,28.879999,Overweight
4,115967096,2009-07-04 11:20:00,33720374,28.878704,NaN,28.878704,Overweight


BMI Summary: {'mean_bmi': 26.91046461743333, 'median_bmi': 25.768348104140212, 'std_bmi': 6.353786127194333, 'min_bmi': 10.014151753272976, 'max_bmi': 96.69268118657513, 'total_measurements': 182089, 'unique_subjects': 4176}


In [32]:
MIN_TESTS = 5
MIN_DAYS_BETWEEN = 0

cbc_df = process_cbc(df[df['subject_id'].isin(bmi_df['subject_id'])], demo_df, MIN_TESTS, MIN_DAYS_BETWEEN)
display(cbc_df.head())
summary_per_subject = get_cbc_subject_statistics(cbc_df)
display(summary_per_subject.head())

# Save CBC
cbc_df.to_csv('data/cbc.csv', index=False)


Extracted 1121726 CBC test results
Aggregated duplicates: 1121244 (3929 subjects) -> 1010036 records (3928 subjects) for 9 tests
After filtering for multiple measurements: 1010036 (3928 subjects) -> 1007368 records (3641 subjects)


,subject_id,code,time,unit,numeric_value,Age,Ethnicity,Gender,Race,within_reference_interval
0,115967096,HCT,2009-07-04 13:15:00,%,39.500000,84,Not Hispanic,F,5,True
1,115967096,HCT,2009-07-26 19:59:00,%,40.500000,84,Not Hispanic,F,5,True
2,115967096,HCT,2009-07-27 05:40:00,%,36.099998,84,Not Hispanic,F,5,True
3,115967096,HCT,2010-06-27 15:45:00,%,35.299999,84,Not Hispanic,F,5,False
4,115967096,HCT,2010-08-22 11:00:00,%,38.700001,84,Not Hispanic,F,5,True


,subject_id,code,num_tests_taken,num_tests_within_reference_interval,percentage_tests_within_reference_interval,days_between_first_and_last,avg_days_between_tests,min_days_between_tests,max_days_between_tests
13951,115970369,MCV,130,87,66.92%,8847,68.59,0.05,860.95
13952,115970369,PLT,132,102,77.27%,8847,67.54,0.05,860.95
13947,115970369,HCT,135,47,34.81%,8847,66.03,0.05,860.95
13948,115970369,HGB,130,53,40.77%,8847,68.59,0.05,860.95
13949,115970369,MCH,130,98,75.38%,8847,68.59,0.05,860.95
